<a href="https://colab.research.google.com/github/JeabtiE/burn-severity-pipeline/blob/main/burning_recurrence_risk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Stage 0


In [11]:
import requests
from google.colab import userdata

# อ่านคีย์จาก Colab Secrets (ไอคอนกุญแจ) ไม่เขียนคีย์ไว้ในโค้ด
map_key = userdata.get("FIRMS_MAP_KEY").strip()
bbox = "98.0,18.2,99.5,20.0"

# Area API ย้อนหลังได้ครั้งละไม่เกิน 5 วัน ใช้แค่ทดสอบคีย์
# ข้อมูลหลายปีของโปรเจคมาจาก Archive Download ไม่ใช่ cell นี้
# จับ error เองเพื่อไม่ให้ traceback พิมพ์ URL ที่มีคีย์ออกมา
for source in ["VIIRS_SNPP_NRT", "MODIS_NRT"]:
    url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{map_key}/{source}/{bbox}/5/2026-03-01"
    try:
        resp = requests.get(url, timeout=30)
        print(source, resp.status_code, "จำนวนบรรทัด:", resp.text.count("\n"))
    except requests.RequestException as e:
        print(source, "เรียก API ไม่สำเร็จ:", type(e).__name__)

# เช็กโควตาคีย์ (ตัวนี้คือการทดสอบจริงว่าคีย์ใหม่ใช้ได้)
try:
    status = requests.get("https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/",
                          params={"MAP_KEY": map_key}, timeout=30)
    print(status.text)
except requests.RequestException as e:
    print("เช็กโควตาไม่สำเร็จ:", type(e).__name__)

VIIRS_SNPP_NRT 200 จำนวนบรรทัด: 0
MODIS_NRT 200 จำนวนบรรทัด: 0
{ "transaction_limit" : 5000, "current_transactions": 0, "transaction_interval" : "10 minutes" }


In [ ]:
!pip install h3 --quiet
import glob
import numpy as np
import pandas as pd
import h3

# path โฟลเดอร์ที่เก็บไฟล์ CSV จาก FIRMS (ปรับตามที่ดาวน์โหลดจริง)
FIRMS_DIR = "/content/drive/MyDrive/burn_severity_pipeline/data/firms"

# ค่าคงที่ที่ต้องล็อกไว้ก่อนดูผล ตามที่ตกลงกันไว้
FIRE_SEASON_MONTHS = (1, 5)       # ม.ค.-พ.ค. เท่านั้น นอกช่วงนี้ตัดทิ้ง
MIN_CONFIDENCE = 50               # ทั้ง MODIS (0-100) และ VIIRS (แปลงเป็นตัวเลขแล้ว) ใช้เกณฑ์เดียวกัน
H3_RESOLUTIONS = [7, 8]           # เทียบสองระดับตามแผน
TOP_SHARE = 0.10                  # "top 10% ของ cell" ตามที่นิยามไว้
DECISION_GATE_MULTIPLIER = 3      # เกณฑ์ผ่าน Stage 0: top 10% ต้องจับ hotspot ได้อย่างน้อย 3 เท่าของสุ่ม (ปรับได้ แต่ต้องล็อกก่อนเห็นผล)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.5 MB/s eta 0:00:00


In [ ]:
def normalize_confidence(df):
    # MODIS: confidence เป็นตัวเลข 0-100 อยู่แล้ว ใช้ตรงๆ
    # VIIRS: confidence เป็นตัวอักษร l/n/h (low/nominal/high) ต้องแปลงเป็นตัวเลขก่อน
    # ค่าที่ map นี้เป็นค่าประมาณคร่าวๆ ไม่ใช่มาตรฐานทางการ ถ้าอยากเข้มงวดกว่านี้
    # ปรับเป็นกรอง l ทิ้งไปเลยก็ได้ แทนการแปลงเป็นตัวเลข
    conf = df["confidence"]
    if conf.dtype == object:
        mapping = {"l": 30, "n": 60, "h": 90}
        return conf.map(mapping).fillna(0)
    return pd.to_numeric(conf, errors="coerce").fillna(0)


def load_hotspots(folder):
    paths = glob.glob(f"{folder}/**/*.csv", recursive=True)
    print("พบไฟล์:", len(paths))
    frames = []
    for p in paths:
        d = pd.read_csv(p)
        d["source_file"] = p.split("/")[-1]
        # MODIS มีคอลัมน์ type (0=vegetation fire, 1=volcano, 2=static land source, 3=offshore)
        # เอาเฉพาะไฟพืชพรรณ ถ้าไม่มีคอลัมน์นี้ (เช่นไฟล์ VIIRS) ก็ไม่ต้องกรอง
        if "type" in d.columns:
            before = len(d)
            d = d[d["type"] == 0]
            print(f"  {p.split('/')[-1]}: ตัด non-vegetation fire ออก {before - len(d)} แถว")
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    df["acq_date"] = pd.to_datetime(df["acq_date"])
    df["confidence_num"] = normalize_confidence(df)
    print("รวมทั้งหมด:", len(df), "แถว")
    print("ช่วงวันที่:", df["acq_date"].min(), "ถึง", df["acq_date"].max())
    print(df.groupby(df["acq_date"].dt.year).size().tail(10))  # เช็คจำนวนต่อปี ปีล่าสุด 10 ปี
    return df


raw = load_hotspots(FIRMS_DIR)

พบไฟล์: 7
  fire_archive_J1V-C2_814332.csv: ตัด non-vegetation fire ออก 20 แถว
  fire_archive_SV-C2_814334.csv: ตัด non-vegetation fire ออก 26 แถว
  fire_archive_M-C61_814331.csv: ตัด non-vegetation fire ออก 331 แถว
รวมทั้งหมด: 617017 แถว
ช่วงวันที่: 2000-12-02 00:00:00 ถึง 2026-09-27 00:00:00
acq_date
2017    17997
2018    15793
2019    67813
2020    73074
2021    36720
2022    14162
2023    50185
2024    70554
2025    29307
2026    62180
dtype: int64


In [ ]:
import os
import glob
import pandas as pd

# ชื่อโฟลเดอร์บอกเซนเซอร์ เช่น DL_FIRE_M-C61_814331 -> รหัส M-C61
SENSOR_BY_TAG = {
    "M-C61": "MODIS",
    "SV-C2": "VIIRS_SNPP",   # Suomi NPP
    "J1V-C2": "VIIRS_N20",   # JPSS-1 = NOAA-20
    "J2V-C2": "VIIRS_N21",   # JPSS-2 = NOAA-21
}


def confidence_to_num(s):
    # MODIS: เป็นตัวเลข 0-100 อยู่แล้ว
    # VIIRS: เป็นตัวอักษร l / n / h (low / nominal / high) แปลงเป็นตัวเลขคร่าวๆ
    # ค่า 30/60/90 เป็นค่าประมาณของผมเอง ไม่ใช่มาตรฐานทางการ ผลคือ l ถูกตัดทิ้งที่เกณฑ์ 50
    # ต้องเรียกทีละไฟล์ก่อนรวม ไม่งั้นตัวเลขกับตัวอักษรจะปนกันในคอลัมน์เดียว
    num = pd.to_numeric(s, errors="coerce")
    letters = s.astype(str).str.lower().map({"l": 30, "n": 60, "h": 90})
    return num.fillna(letters).fillna(0)


def load_hotspots(folder):
    paths = sorted(glob.glob(f"{folder}/**/*.csv", recursive=True))  # ค้นในโฟลเดอร์ย่อยด้วย
    print("พบไฟล์:", len(paths))
    keep = ["latitude", "longitude", "acq_date", "acq_time", "confidence", "frp", "daynight"]
    frames = []
    for p in paths:
        name = os.path.basename(p)
        tag = os.path.basename(os.path.dirname(p)).split("_")[2]
        d = pd.read_csv(p)
        n0 = len(d)
        if "type" in d.columns:
            # 0 = ไฟบนพืชพรรณ (ตัดภูเขาไฟ, แหล่งความร้อนคงที่, จุดในทะเลออก)
            d = d[d["type"] == 0]
        d = d[[c for c in keep if c in d.columns]].copy()
        d["confidence_num"] = confidence_to_num(d["confidence"])
        d["sensor"] = SENSOR_BY_TAG.get(tag, tag)
        d["product"] = "nrt" if "nrt" in name else "archive"
        print(f"  {name}: {n0} แถว -> ใช้ {len(d)} แถว")
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    df["acq_date"] = pd.to_datetime(df["acq_date"])
    # กันข้อมูลซ้ำตรงรอยต่อของไฟล์ archive กับ nrt
    before = len(df)
    df = df.drop_duplicates(subset=["latitude", "longitude", "acq_date", "acq_time", "sensor"])
    print(f"ตัดแถวซ้ำ {before - len(df)} แถว เหลือรวม {len(df)} แถว")
    return df


raw = load_hotspots(FIRMS_DIR)
print("ช่วงวันที่:", raw["acq_date"].min().date(), "ถึง", raw["acq_date"].max().date())

# จำนวน hotspot ต่อปี แยกตามเซนเซอร์ (ทุกเดือน) ดูว่าแต่ละเซนเซอร์เริ่มมีข้อมูลปีไหน
by_year = raw.pivot_table(index=raw["acq_date"].dt.year, columns="sensor",
                          values="latitude", aggfunc="count").fillna(0).astype(int)
print(by_year)

พบไฟล์: 7
  fire_archive_J1V-C2_814332.csv: 172595 แถว -> ใช้ 172575 แถว
  fire_nrt_J1V-C2_814332.csv: 10 แถว -> ใช้ 10 แถว
  fire_nrt_J2V-C2_814333.csv: 47853 แถว -> ใช้ 47853 แถว
  fire_archive_M-C61_814331.csv: 74555 แถว -> ใช้ 74224 แถว
  fire_nrt_M-C61_814331.csv: 1 แถว -> ใช้ 1 แถว
  fire_archive_SV-C2_814334.csv: 322371 แถว -> ใช้ 322345 แถว
  fire_nrt_SV-C2_814334.csv: 9 แถว -> ใช้ 9 แถว
ตัดแถวซ้ำ 0 แถว เหลือรวม 617017 แถว
ช่วงวันที่: 2000-12-02 ถึง 2026-09-27
sensor    MODIS  VIIRS_N20  VIIRS_N21  VIIRS_SNPP
acq_date                                         
2000         11          0          0           0
2001        808          0          0           0
2002        717          0          0           0
2003       2663          0          0           0
2004       5380          0          0           0
2005       3302          0          0           0
2006       2873          0          0           0
2007       5012          0          0           0
2008       2648          0 

In [ ]:
!ls -la /content/drive/MyDrive/burn_severity_pipeline/recurrence/
!find /content/drive/MyDrive -iname "stage0*"

total 26
drwx------ 2 root root  4096 Oct  1 06:32 .
drwx------ 3 root root  4096 Sep 27 10:41 ..
-rw------- 1 root root 18297 Oct  1 06:32 stage0.py
/content/drive/MyDrive/burn_severity_pipeline/recurrence/stage0.py


In [ ]:
!pip install h3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.9 MB/s eta 0:00:00


In [ ]:
import sys, importlib
importlib.invalidate_caches()
p = "/content/drive/MyDrive/burn_severity_pipeline/recurrence"
if p not in sys.path:
    sys.path.append(p)
import stage0

FIRMS_DIR = "/content/drive/MyDrive/burn_severity_pipeline/data/firms"
OUT_DIR = "/content/drive/MyDrive/burn_severity_pipeline/results/stage0"
summary = stage0.run_all(FIRMS_DIR, OUT_DIR)

พบไฟล์: 7
  fire_archive_J1V-C2_814332.csv: 172595 แถว -> ใช้ 172575 แถว
  fire_nrt_J1V-C2_814332.csv: 10 แถว -> ใช้ 10 แถว
  fire_nrt_J2V-C2_814333.csv: 47853 แถว -> ใช้ 47853 แถว
  fire_archive_M-C61_814331.csv: 74555 แถว -> ใช้ 74224 แถว
  fire_nrt_M-C61_814331.csv: 1 แถว -> ใช้ 1 แถว
  fire_archive_SV-C2_814334.csv: 322371 แถว -> ใช้ 322345 แถว
  fire_nrt_SV-C2_814334.csv: 9 แถว -> ใช้ 9 แถว
ตัดแถวซ้ำ 0 แถว เหลือรวม 617017 แถว

######## A_SNPP: VIIRS_SNPP ฤดู 2012-2026 (279714 hotspot หลังกรอง) ########
season
2012    22356
2013    19815
2014    20080
2015    23478
2016    23900
2017    13751
2018     9685
2019    28281
2020    30903
2021    15473
2022     5780
2023    20286
2024    20937
2025     7618
2026    17371

=== A_SNPP H3 resolution 7 ===
จำนวน cell ในจักรวาล: 5442

-- recurrence รายฤดู (rel_risk > 1 = ไหม้ซ้ำมากกว่าสุ่ม) --
 season  prev_area_share  p_if_prev  p_if_not  rel_risk  hot_share_in_prev  hot_lift
   2013            0.787      0.835     0.410     2.037          

# Stage 1
- terrain_export.py : ส่งออกภูมิประเทศ SRTM สำหรับ Stage 1 (ตาม stage1_prereg.md ข้อ 4)
- รันใน Colab ทีละ cell (แต่ละ cell คั่นด้วย # %%)
- ผลลัพธ์: GeoTIFF 2 แถบ (elevation, slope) ความละเอียด 90 ม. EPSG:32647
- เส้นตายตาม prereg: ไฟล์ต้องอยู่ใน Drive ภายในสิ้นวันที่ 4 ต.ค. 2026

In [2]:
# %% cell 1: เชื่อม Earth Engine
import ee
ee.Authenticate()
ee.Initialize(project="burn-severity-pipeline")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [3]:
# %% cell 2: เตรียมภาพและสั่ง export
BBOX = (98.0, 18.2, 99.5, 20.0)   # west, south, east, north ต้องตรงกับ stage0.BBOX

# geodesic=False ให้ขอบสี่เหลี่ยมเป็นเส้นตรงในพิกัด lon/lat เหมือนกรอบที่ขอจาก FIRMS
region = ee.Geometry.Rectangle(list(BBOX), proj="EPSG:4326", geodesic=False)

# SRTM 1 arc-second (ราว 30 ม.) แถบ elevation หน่วยเมตร
dem = ee.Image("USGS/SRTMGL1_003").select("elevation")

# ความชันคำนวณที่ความละเอียดเดิม 30 ม. ก่อน (หน่วยองศา)
# ถ้าย่อเป็น 90 ม. ก่อนแล้วค่อยคำนวณ ความชันจะต่ำกว่าจริงเพราะภูมิประเทศถูกปรับเรียบไปแล้ว
slope = ee.Terrain.slope(dem)

img = dem.addBands(slope).toFloat()   # แถบ: elevation, slope

# ย่อ 30 ม. -> 90 ม. ด้วยค่าเฉลี่ยจริง (reduceResolution)
# ถ้าใช้ reproject อย่างเดียว Earth Engine จะหยิบค่าแบบ nearest ไม่ใช่ค่าเฉลี่ย
proj90 = ee.Projection("EPSG:32647").atScale(90)
img90 = img.reduceResolution(reducer=ee.Reducer.mean(), maxPixels=64).reproject(crs=proj90)

# ใช้ Export.image.toDrive ไม่ใช้ getDownloadURL เพราะไฟล์ใหญ่เกินเพดานของ getDownloadURL
# folder คือชื่อโฟลเดอร์ระดับบนสุดใน MyDrive (กำหนด path ซ้อนไม่ได้)
task = ee.batch.Export.image.toDrive(
    image=img90,
    description="srtm_terrain_90m",
    folder="burn_severity_terrain",
    fileNamePrefix="srtm_terrain_90m",
    region=region,
    crs="EPSG:32647",
    scale=90,
    maxPixels=1e9,
    fileFormat="GeoTIFF",
)
task.start()
print("สั่ง export แล้ว task id:", task.id)

สั่ง export แล้ว task id: HK3TCIXGJONKFYLL5RCCQ7OC


In [6]:
# %% cell 3: ดูสถานะ (รันซ้ำได้เรื่อยๆ ปกติใช้เวลาหลายนาที)
# READY = รอคิว, RUNNING = กำลังทำ, COMPLETED = เสร็จ, FAILED = ล้มเหลว (ดู error_message)
st = task.status()
print(st["state"], st.get("error_message", ""))

COMPLETED 


In [7]:
# %% cell 4: ตรวจไฟล์หลัง export เสร็จ (ต้อง mount Drive ก่อน)
from google.colab import drive
drive.mount("/content/drive")

import glob
import numpy as np
import rasterio

paths = glob.glob("/content/drive/MyDrive/burn_severity_terrain/srtm_terrain_90m*.tif")
print("พบไฟล์:", paths)   # ถ้าได้ [] ให้รอ Drive ซิงก์สักครู่ หรือเช็กชื่อโฟลเดอร์

# ไฟล์ใหญ่อาจถูกแบ่งเป็นหลายชิ้น (ชื่อลงท้าย -0000000000-0000000000.tif) ให้รายงานจำนวนชิ้นมาด้วย
for p in paths:
    with rasterio.open(p) as src:
        print("\n", p)
        print("ขนาด:", src.width, "x", src.height, "แถบ:", src.count, "crs:", src.crs)
        print("ขนาดพิกเซล:", src.res, "nodata:", src.nodata)
        for b, name in [(1, "elevation"), (2, "slope")]:
            a = src.read(b, masked=True)
            print(f"{name}: min {a.min():.1f} max {a.max():.1f} mean {a.mean():.1f} "
                  f"พิกเซลไม่มีข้อมูล {a.mask.mean():.2%}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
พบไฟล์: ['/content/drive/MyDrive/burn_severity_terrain/srtm_terrain_90m.tif']

 /content/drive/MyDrive/burn_severity_terrain/srtm_terrain_90m.tif
ขนาด: 1764 x 2217 แถบ: 2 crs: EPSG:32647
ขนาดพิกเซล: (90.0, 90.0) nodata: None
elevation: min 150.0 max 2572.0 mean 812.3 พิกเซลไม่มีข้อมูล 0.00%
slope: min 0.0 max 71.7 mean 16.2 พิกเซลไม่มีข้อมูล 0.00%


In [8]:
# colab_stage1.py : ลำดับการรัน Stage 1 ใน Colab (คัดลอกทีละ cell ที่คั่นด้วย # %%)
# ก่อน cell 5 ต้อง commit stage1_prereg.md เข้า GitHub แล้ว

# %% cell 1: เตรียมสภาพแวดล้อม (รันใหม่ทุกครั้งที่ runtime รีสตาร์ท)
from google.colab import drive
drive.mount("/content/drive")
!pip install h3 -q

import sys, glob, os, importlib
ROOT = "/content/drive/MyDrive/burn_severity_pipeline"
sys.path.append(f"{ROOT}/recurrence")
importlib.invalidate_caches()
import stage0, stage1
importlib.reload(stage0)
importlib.reload(stage1)

FIRMS_DIR = f"{ROOT}/data/firms"
STAGE0_DIR = f"{ROOT}/results/stage0"
OUT_DIR = f"{ROOT}/results/stage1"
os.makedirs(OUT_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.2 MB/s eta 0:00:00


In [9]:
# %% cell 2: โหลด hotspot ครั้งเดียว แล้วเตรียมชุดหลัก (A_SNPP res 8)
raw = stage0.load_hotspots(FIRMS_DIR)
data = stage1.prepare(raw, "A_SNPP", 8)

พบไฟล์: 7
  fire_archive_J1V-C2_814332.csv: 172595 แถว -> ใช้ 172575 แถว
  fire_nrt_J1V-C2_814332.csv: 10 แถว -> ใช้ 10 แถว
  fire_nrt_J2V-C2_814333.csv: 47853 แถว -> ใช้ 47853 แถว
  fire_archive_M-C61_814331.csv: 74555 แถว -> ใช้ 74224 แถว
  fire_nrt_M-C61_814331.csv: 1 แถว -> ใช้ 1 แถว
  fire_archive_SV-C2_814334.csv: 322371 แถว -> ใช้ 322345 แถว
  fire_nrt_SV-C2_814334.csv: 9 แถว -> ใช้ 9 แถว
ตัดแถวซ้ำ 0 แถว เหลือรวม 617017 แถว
[A_SNPP res 8] cell 37359, ป้ายฤดู 2015-2026, ทดสอบ 2018-2026 (9 ฤดู) ใช้เวลา 6 วินาที


In [10]:
# %% cell 3: จุดตรวจ 9.3 (ต้องขึ้น "ผ่าน" ถ้าขึ้น AssertionError ให้หยุดแล้ววางผลกลับมา)
chk = stage1.check_b1(data, f"{STAGE0_DIR}/A_SNPP_res8_baselines.csv")

จุดตรวจ 9.3: เทียบ 180 ค่า (12 ฤดู) ผลต่างมากสุด 4.44e-16
ผ่าน: baseline ตรงกับ Stage 0
